<a href="https://colab.research.google.com/github/serenyang2026/ai-agents/blob/main/Session_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Use case**

Implement a session manager for a web application (or an AI agent backend). When a user logs in or starts a conversation, they get a session. The session should expire after a fiexed period of time. Users can alos end their session manually.

**Requirements**



*   create_session: create a new session and return a unique session id
*   get_session: returns the user_id if the session exists and has not expired; otherwise returns None
*   delete_session: removes the session; it should not raise an error if the session doesn't exist
*   every session has the same time-to-live(TTL), set when the manager is created
*   Expired sessions should be removed when they are accessed.





In [30]:
import time
import uuid
import heapq
import threading

class SessionManager:
  def __init__(self, ttl_seconds, cleanup_interval = 1):
    self.ttl = ttl_seconds
    self.sessions = {}
    self.heap =[]
    self.lock = threading.Lock()
    self.cleanup_interval = cleanup_interval
    self.cleanup_thread = threading.Thread(target = self._cleanup_loop, daemon= True)
    self.cleanup_thread.start()


  def create_session(self, user_id):
    session_id = str(uuid.uuid4())
    now = time.time()
    expires_at = now + self.ttl

    with self.lock:

      self.sessions[session_id] = {"user_id": user_id, "expires_at": expires_at}
      heapq.heappush(self.heap, (expires_at, session_id))
      return session_id


  def get_session(self, session_id):
    with self.lock:
      session = self.sessions.get(session_id)
      if session is None:
        return None
      if session["expires_at"] <= time.time():
        self.sessions.pop(session_id)
        return None
      session["expires_at"] = time.time() + self.ttl
      return session["user_id"]


  def delete_session(self, session_id):
    with self.lock:
      self.sessions.pop(session_id, None)

  def cleanup_expired(self):
    now = time.time()
    with self.lock:
      while self.heap and self.heap[0][0] <= now:
        expires_at, session_id = heapq.heappop(self.heap)
        session = self.sessions.get(session_id)
        if session is None:
          continue
        if session["expires_at"] <= now:
          self.sessions.pop(session_id)
        else:
          heapq.heappush(self.heap, (session["expires_at"], session_id))


  def _cleanup_loop(self):
    while True:
        self.cleanup_expired()
        time.sleep(self.cleanup_interval)





Session Manager - Redis

In [ ]:
class SesseionManager:
  def __init__(self, redis_client, ttl_seconds):
    self.redis = redis_client
    self.ttl = ttl_seconds

  def create_session(self, user_id):
    session_id = str(uuid.uuid4())
    self.redis.set(session_id, user_id, ex=self.ttl)
    return session_id

  def get_session(self, session_id):
    user_id = self.redis.get(session_id)
    if user_id is Nonw:
      return None
    self.redis.expire(session_id, self.ttl)
    return user_id

  def delete_session(self, session_id):
    self.redis.delete(session_id)
